# ADAPT-ECG — Fase 3B: Reentrenamiento Continuo (ResNet-SE + Replay Buffer)

**Objetivo:** Demostrar que el modelo base `ECG_ResNet_SE` mejora progresivamente
al recibir datos de nuevos pacientes mediante aprendizaje incremental con Replay Buffer.

**Concepto:**
- El dataset se divide en 6 lotes simulando llegada temporal de pacientes
- Lote 0 = datos de entrenamiento original del modelo base
- Lotes 1-5 = nuevos pacientes que llegan en el tiempo
- Modelo **estático**: nunca aprende de datos nuevos
- Modelo **adaptativo**: reentrena en cada lote (50% nuevos + 50% buffer)

**Archivos necesarios en Drive (`ADAPT-ECG/`):**
- `X_128.npy`, `y_128.npy` — dataset completo con ventana 128
- `ecg_resnet_se_base.pth` — modelo base entrenado en Fase 2B

**Tiempo estimado:** ~15 minutos con GPU T4

---
## CELDA 1 — GPU + dependencias

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

---
## CELDA 2 — Montar Drive y cargar datos + modelo base

In [ ]:
from google.colab import drive
import numpy as np

drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/'  # ajusta si tus archivos estan en subcarpeta

X = np.load(DRIVE_DIR + 'X_128.npy')
y = np.load(DRIVE_DIR + 'y_128.npy')

print(f'Dataset: {X.shape}  |  Clases: {np.unique(y, return_counts=True)}')

---
## CELDA 3 — Definir arquitectura ECG_ResNet_SE

Debe ser identica a la usada en Fase 2B para cargar correctamente los pesos.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class ResBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=5):
        super().__init__()
        pad = kernel // 2
        self.conv1    = nn.Conv1d(in_ch, out_ch, kernel, padding=pad, bias=False)
        self.bn1      = nn.BatchNorm1d(out_ch)
        self.conv2    = nn.Conv1d(out_ch, out_ch, kernel, padding=pad, bias=False)
        self.bn2      = nn.BatchNorm1d(out_ch)
        self.shortcut = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, 1, bias=False),
            nn.BatchNorm1d(out_ch)
        ) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        residual = self.shortcut(x)
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        return F.relu(x + residual)


class SEBlock1D(nn.Module):
    def __init__(self, channels, ratio=8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc   = nn.Sequential(
            nn.Linear(channels, channels // ratio),
            nn.ReLU(),
            nn.Linear(channels // ratio, channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.shape
        s = self.pool(x).squeeze(-1)
        e = self.fc(s).unsqueeze(-1)
        return x * e


class ECG_ResNet_SE(nn.Module):
    def __init__(self, n_classes=5, input_len=128):
        super().__init__()
        self.stem   = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=7, padding=3, bias=False),
            nn.BatchNorm1d(32), nn.ReLU())
        self.layer1 = nn.Sequential(ResBlock1D(32,  32),  nn.MaxPool1d(2))
        self.layer2 = nn.Sequential(ResBlock1D(32,  64),  nn.MaxPool1d(2))
        self.layer3 = nn.Sequential(ResBlock1D(64, 128),  nn.MaxPool1d(2))
        self.se     = SEBlock1D(128, ratio=8)
        self.pool   = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(64, n_classes))

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x); x = self.layer2(x); x = self.layer3(x)
        x = self.se(x);     x = self.pool(x)
        return self.classifier(x)


print('Arquitectura definida OK')

---
## CELDA 4 — Cargar modelo base y crear modelo estatico (copia congelada)

El modelo **estático** nunca se actualiza — es la línea base de comparación.  
El modelo **adaptativo** parte del mismo punto y aprende en cada lote.

In [ ]:
import copy

# Cargar modelo base
modelo_base = ECG_ResNet_SE(n_classes=5).to(device)
modelo_base.load_state_dict(
    torch.load(DRIVE_DIR + 'ecg_resnet_se_base.pth', map_location=device)
)
modelo_base.eval()

# Modelo estatico: copia que NUNCA se modifica
modelo_estatico = copy.deepcopy(modelo_base)
modelo_estatico.eval()

# Modelo adaptativo: empieza igual, se actualiza en cada lote
modelo_adaptativo = copy.deepcopy(modelo_base)

print('Modelo base cargado correctamente')
print(f'Parametros: {sum(p.numel() for p in modelo_base.parameters()):,}')

---
## CELDA 5 — Replay Buffer y funciones de evaluacion

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score

# ---------------------------------------------------------------------------
# Replay Buffer FIFO
# ---------------------------------------------------------------------------
class ReplayBuffer:
    def __init__(self, max_size=2000):
        self.X        = []
        self.y        = []
        self.max_size = max_size

    def add(self, X_batch, y_batch):
        self.X.extend(X_batch.tolist())
        self.y.extend(y_batch.tolist())
        if len(self.X) > self.max_size:
            self.X = self.X[-self.max_size:]
            self.y = self.y[-self.max_size:]

    def sample(self, n):
        idx = np.random.choice(len(self.X), size=min(n, len(self.X)), replace=False)
        return (np.array([self.X[i] for i in idx], dtype=np.float32),
                np.array([self.y[i] for i in idx], dtype=np.int64))

    def save(self, path):
        np.savez(path, X=np.array(self.X, dtype=np.float32),
                       y=np.array(self.y, dtype=np.int64))

    def __len__(self):
        return len(self.X)


# ---------------------------------------------------------------------------
# Evaluacion rapida
# ---------------------------------------------------------------------------
def evaluar(model, X_np, y_np, batch_size=256):
    model.eval()
    X_t  = torch.tensor(X_np, dtype=torch.float32).unsqueeze(1)
    loader = DataLoader(TensorDataset(X_t), batch_size=batch_size, shuffle=False)
    preds = []
    with torch.no_grad():
        for (xb,) in loader:
            preds.extend(model(xb.to(device)).argmax(1).cpu().numpy())
    preds = np.array(preds)
    acc = accuracy_score(y_np, preds)
    f1  = f1_score(y_np, preds, average='macro', zero_division=0)
    return acc, f1, preds


# ---------------------------------------------------------------------------
# Reentrenamiento incremental con Replay Buffer
# ---------------------------------------------------------------------------
def reentrenar(model, X_new, y_new, buffer, epochs=5, lr=1e-4, batch_size=64):
    """
    50% datos nuevos + 50% muestras del buffer.
    Loss con pesos de clase para mantener atencion en clases raras.
    """
    # Calcular pesos de clase en los datos nuevos
    clases, conteos = np.unique(y_new, return_counts=True)
    pesos = np.ones(5, dtype=np.float32)
    for c, cnt in zip(clases, conteos):
        pesos[c] = 1.0 / np.sqrt(cnt + 1)
    pesos = pesos / pesos.sum() * 5
    criterion = nn.CrossEntropyLoss(
        weight=torch.tensor(pesos).to(device)
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    model.train()

    # Agregar al buffer antes de entrenar
    buffer.add(X_new, y_new)

    for _ in range(epochs):
        n_replay = len(X_new)
        if len(buffer) > 0 and n_replay > 0:
            Xr, yr  = buffer.sample(n_replay)
            X_comb  = np.concatenate([X_new, Xr])
            y_comb  = np.concatenate([y_new, yr])
        else:
            X_comb, y_comb = X_new, y_new

        perm   = np.random.permutation(len(X_comb))
        Xt = torch.tensor(X_comb[perm], dtype=torch.float32).unsqueeze(1)
        yt = torch.tensor(y_comb[perm], dtype=torch.long)
        loader = DataLoader(TensorDataset(Xt, yt), batch_size=batch_size, shuffle=True)

        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

    model.eval()


print('ReplayBuffer y funciones definidas OK')

---
## CELDA 6 — Dividir dataset en lotes temporales (simula llegada de pacientes)

6 lotes iguales:
- **Lote 0**: representa el conjunto con que se entrenó el modelo base
- **Lotes 1–5**: nuevos pacientes que llegan en el tiempo

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

N_LOTES   = 6
CLASES    = ['N', 'S', 'V', 'F', 'Q']
SEED      = 42
np.random.seed(SEED)

# Shuffle estratificado y dividir en N_LOTES partes iguales
idx = np.arange(len(y))
np.random.shuffle(idx)
lotes_idx = np.array_split(idx, N_LOTES)

print(f'Dataset dividido en {N_LOTES} lotes:')
for i, lote in enumerate(lotes_idx):
    clases_lote = {c: int((y[lote]==j).sum()) for j, c in enumerate(CLASES)}
    print(f'  Lote {i}: {len(lote):,} latidos  |  {clases_lote}')

---
## CELDA 7 — Bucle de reentrenamiento continuo

En cada lote:
1. Evalúa modelo **estático** (nunca cambia)
2. Evalúa modelo **adaptativo** ANTES de reentrenar
3. Reentrena el adaptativo con Replay Buffer
4. Evalúa adaptativo DESPUÉS de reentrenar
5. Registra todas las métricas

In [ ]:
# Hiperparametros de reentrenamiento
EPOCHS_PER_BATCH = 5
LR_RETRAIN       = 1e-4
BUFFER_MAX_SIZE  = 2000

buffer    = ReplayBuffer(max_size=BUFFER_MAX_SIZE)
resultados = []

print(f'{'Lote':>5}  {'N_beats':>8}  {'Acc_Est':>8}  {'F1_Est':>7}  '
      f'{'Acc_Ant':>8}  {'F1_Ant':>7}  {'Acc_Dep':>8}  {'F1_Dep':>7}  {'Buffer':>7}')
print('-' * 85)

for lote_num, lote_idx in enumerate(lotes_idx):
    X_lote = X[lote_idx]
    y_lote = y[lote_idx]

    # 1. Evaluar estatico
    acc_est, f1_est, _ = evaluar(modelo_estatico, X_lote, y_lote)

    # 2. Evaluar adaptativo ANTES
    acc_antes, f1_antes, _ = evaluar(modelo_adaptativo, X_lote, y_lote)

    # 3. Reentrenar adaptativo (lote 0 inicializa el buffer sin reentrenar)
    if lote_num == 0:
        buffer.add(X_lote, y_lote)
        acc_despues, f1_despues = acc_antes, f1_antes
    else:
        reentrenar(modelo_adaptativo, X_lote, y_lote, buffer,
                   epochs=EPOCHS_PER_BATCH, lr=LR_RETRAIN)
        # 4. Evaluar adaptativo DESPUES
        acc_despues, f1_despues, _ = evaluar(modelo_adaptativo, X_lote, y_lote)

    resultados.append({
        'lote'        : lote_num,
        'n_beats'     : len(lote_idx),
        'acc_estatico': acc_est,
        'f1_estatico' : f1_est,
        'acc_antes'   : acc_antes,
        'f1_antes'    : f1_antes,
        'acc_despues' : acc_despues,
        'f1_despues'  : f1_despues,
        'buffer_size' : len(buffer),
    })

    print(f'{lote_num:>5}  {len(lote_idx):>8,}  {acc_est:>8.4f}  {f1_est:>7.4f}  '
          f'{acc_antes:>8.4f}  {f1_antes:>7.4f}  {acc_despues:>8.4f}  {f1_despues:>7.4f}  '
          f'{len(buffer):>7,}')

print('\nReentrenamiento continuo completado.')

---
## CELDA 8 — Evaluacion final completa: estatico vs adaptativo

In [ ]:
from sklearn.metrics import classification_report
from scipy.stats import chi2

# Evaluar en TODO el dataset
acc_est_total,  f1_est_total,  preds_est  = evaluar(modelo_estatico,   X, y)
acc_adap_total, f1_adap_total, preds_adap = evaluar(modelo_adaptativo, X, y)

print('MODELO ESTATICO — dataset completo')
print(classification_report(y, preds_est, target_names=['N','S','V','F','Q'], digits=4))

print('MODELO ADAPTATIVO — dataset completo')
print(classification_report(y, preds_adap, target_names=['N','S','V','F','Q'], digits=4))

# McNemar manual
correcto_est  = (preds_est  == y)
correcto_adap = (preds_adap == y)
b = int(((~correcto_est)  & correcto_adap).sum())  # est falla, adap acierta
c = int((correcto_est  & (~correcto_adap)).sum())   # est acierta, adap falla
if (b + c) > 0:
    chi2_stat = (abs(b - c) - 1) ** 2 / (b + c)
    p_val     = 1 - chi2.cdf(chi2_stat, df=1)
else:
    chi2_stat, p_val = 0.0, 1.0

print('RESUMEN COMPARATIVO')
print(f'  Accuracy  Estatico: {acc_est_total:.4f}   Adaptativo: {acc_adap_total:.4f}   Delta: {acc_adap_total-acc_est_total:+.4f}')
print(f'  F1 Macro  Estatico: {f1_est_total:.4f}   Adaptativo: {f1_adap_total:.4f}   Delta: {f1_adap_total-f1_est_total:+.4f}')
print(f'  McNemar: chi2={chi2_stat:.2f}  p={p_val:.4e}')
if p_val < 0.05:
    print('  -> Diferencia estadisticamente significativa (p < 0.05)')

---
## CELDA 9 — Graficas: evolucion por lote + barras de comparacion

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

lotes     = [r['lote']         for r in resultados]
f1_est    = [r['f1_estatico']  for r in resultados]
f1_adap   = [r['f1_despues']   for r in resultados]
acc_est   = [r['acc_estatico'] for r in resultados]
acc_adap  = [r['acc_despues']  for r in resultados]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# F1 Macro por lote
axes[0].plot(lotes, f1_est,  'o--', color='#607D8B', label='Estatico',    linewidth=2)
axes[0].plot(lotes, f1_adap, 's-',  color='#2196F3', label='Adaptativo',  linewidth=2)
axes[0].fill_between(lotes, f1_est, f1_adap, alpha=0.15, color='#2196F3')
axes[0].set_title('F1 Macro por lote temporal', fontsize=13)
axes[0].set_xlabel('Lote (tiempo)')
axes[0].set_ylabel('F1 Macro')
axes[0].set_ylim([0.7, 1.0])
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_xticks(lotes)
axes[0].set_xticklabels([f'Lote {i}' for i in lotes], rotation=20)

# Accuracy por lote
axes[1].plot(lotes, acc_est,  'o--', color='#607D8B', label='Estatico',   linewidth=2)
axes[1].plot(lotes, acc_adap, 's-',  color='#4CAF50', label='Adaptativo', linewidth=2)
axes[1].fill_between(lotes, acc_est, acc_adap, alpha=0.15, color='#4CAF50')
axes[1].set_title('Accuracy por lote temporal', fontsize=13)
axes[1].set_xlabel('Lote (tiempo)')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim([0.88, 1.0])
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_xticks(lotes)
axes[1].set_xticklabels([f'Lote {i}' for i in lotes], rotation=20)

plt.suptitle('ADAPT-ECG Fase 3B — Reentrenamiento Continuo (ResNet-SE)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('/content/fase3b_evolucion.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafica guardada en /content/fase3b_evolucion.png')

---
## CELDA 10 — Guardar modelo adaptativo + buffer + resultados

In [ ]:
import json
from datetime import datetime
from google.colab import files

# Guardar modelo adaptativo
torch.save(modelo_adaptativo.state_dict(), '/content/ecg_resnet_se_retrained.pth')
torch.save(modelo_adaptativo.state_dict(), DRIVE_DIR + 'ecg_resnet_se_retrained.pth')

# Guardar buffer de replay
buffer.save('/content/replay_buffer_128.npz')
buffer.save(DRIVE_DIR + 'replay_buffer_128.npz')

# Guardar resultados JSON
resumen = {
    'fecha'           : datetime.now().strftime('%Y-%m-%d %H:%M'),
    'modelo'          : 'ECG_ResNet_SE',
    'beat_len'        : 128,
    'n_lotes'         : N_LOTES,
    'epochs_per_batch': EPOCHS_PER_BATCH,
    'lr_retrain'      : LR_RETRAIN,
    'buffer_max_size' : BUFFER_MAX_SIZE,
    'estatico' : {
        'accuracy': round(acc_est_total,  4),
        'f1_macro': round(f1_est_total,   4)
    },
    'adaptativo': {
        'accuracy': round(acc_adap_total, 4),
        'f1_macro': round(f1_adap_total,  4)
    },
    'mejora': {
        'accuracy': round(acc_adap_total - acc_est_total, 4),
        'f1_macro': round(f1_adap_total  - f1_est_total,  4)
    },
    'mcnemar': {
        'chi2'  : round(chi2_stat, 2),
        'p_valor': float(p_val)
    },
    'resultados_por_lote': resultados
}

with open('/content/fase3b_resultados.json', 'w') as f:
    json.dump(resumen, f, indent=2, ensure_ascii=False)

with open(DRIVE_DIR + 'fase3b_resultados.json', 'w') as f:
    json.dump(resumen, f, indent=2, ensure_ascii=False)

print('Archivos guardados en Drive y descargando a tu PC...')
files.download('/content/ecg_resnet_se_retrained.pth')
files.download('/content/replay_buffer_128.npz')
files.download('/content/fase3b_resultados.json')
files.download('/content/fase3b_evolucion.png')

print('\nRESUMEN:')
print(json.dumps({k: v for k, v in resumen.items() if k != 'resultados_por_lote'},
                 indent=2, ensure_ascii=False))